In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb  
from sklearn.preprocessing import PolynomialFeatures,StandardScaler
from sklearn.model_selection import train_test_split  #分割数据集
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error   #模型评价
import warnings
warnings.filterwarnings("ignore")

In [ ]:
###模型训练

In [ ]:
X_train,X_test= train_test_split(X, test_size=0.2,random_state =2)
y_train,y_test = train_test_split(y, test_size=0.2,random_state = 2)
from sklearn.preprocessing  import StandardScaler
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(n_estimators=91,max_depth=7,random_state=0,alpha=0.1)
model.fit(X_train_std, y_train)
y_train_pred = model.predict(X_train_std)
y_test_pred = model.predict(X_test_std)
score=r2_score(y_test,y_test_pred)

MSE_train = mean_squared_error(y_train,y_train_pred) 
MSE_test = mean_squared_error(y_test,y_test_pred)
MAE_train = mean_absolute_error(y_train,y_train_pred) 
MAE_test= mean_absolute_error(y_test,y_test_pred)
print('MSE_train:',MSE_train,'\n','MSE_test',MSE_test) 
print('MAE_train:',MAE_train,'\n','MAE_test',MAE_test)
print("R^2_train:",r2_score(y_train,y_train_pred),'\n',"R^2_test1:",r2_score(y_test,y_test_pred))

plt.figure(figsize=(5,3)) 
plt.scatter(y_train,y_train_pred,color='red',marker='d',edgecolor='white',label ='Training data') 
plt.scatter(y_test,y_test_pred,color='blue',marker='s',edgecolor='white',label ='Test data')
plt.ylabel('predicted ')
plt.xlabel('Experimental ')
plt.title('Comparison between the experimental and predicted ')
plt.legend(loc = 'upper left') #图例位置
plt.plot([0.3, 1.8], [0.3,1.8],c='black')
plt.xlim([0.3,1.8]) # 设定x轴范围
plt.ylim(0.3,1.8)
plt.tight_layout() 

plt.show()

In [ ]:
##SHAP分析
sc = StandardScaler()
X= sc.fit_transform(X)
X1=df.iloc[:,:]
X=pd.DataFrame(X,columns=X1.columns)
import shap
import json 
shap.initjs()
explainer = shap.Explainer(model,X)
shap_values = explainer(X)
shap.summary_plot(shap_values, X,max_display=10)
shap.summary_plot(shap_values, X,max_display=10, 
                  plot_type="bar")
import matplotlib.pyplot as plt
element_names=X.columns.values[:]
shap_values=shap_values.values
shap_values=pd.DataFrame(shap_values,columns=element_names)
#fig, ax = plt.subplots()
plt.figure(figsize=(3,4)) #
feature_name='Hs_wt'
shap_value=shap_values[feature_name]
colors = ['red' if val > 0 else 'blue' for val in shap_value]
a=df[feature_name]

plt.scatter(a,  
            shap_value, 
            c=colors ,s=20)
plt.axvline(x=0.35, color='black', linestyle='--', linewidth=3)
plt.xlabel(feature_name)
plt.ylabel('SHAP value')
plt.title("SHAP value vs. Feature Value")
plt.show()

In [ ]:
###不同数据量

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

ratios = [0.6, 0.7, 0.8]  


random_states = np.arange(1, 21, 2)
learning_rates = np.arange(0.01, 0.5, 0.05)
n_estimators_list = np.arange(10, 100, 10)
max_depth_list = np.arange(3, 15, 2)

print("="*50)
print("不同数据量对模型性能的影响（含超参数搜索）")
print("="*50)

for ratio in ratios:
    # 抽取子数据集
    X_subset, _, y_subset, _ = train_test_split(X, y, train_size=ratio, random_state=0)
    
    best_score = -np.inf
    best_parameters = {}

    for rs in random_states:
        for lr in learning_rates:
            for n_est in n_estimators_list:
                for depth in max_depth_list:
                    # 固定随机划分训练/测试集
                    X_tr, X_te, y_tr, y_te = train_test_split(
                        X_subset, y_subset, test_size=0.2, random_state=rs
                    )
                    
                    # 标准化
                    sc = StandardScaler()
                    X_tr_std = sc.fit_transform(X_tr)
                    X_te_std = sc.transform(X_te)
                    
                    # 模型训练
                    model = GradientBoostingRegressor(
                        learning_rate=lr,
                        n_estimators=n_est,
                        max_depth=depth,
                        random_state=0
                    )
                    model.fit(X_tr_std, y_tr)
                    
                    # 测试集预测
                    y_te_pred = model.predict(X_te_std)
                    score = r2_score(y_te, y_te_pred)
                    
                    # 保存最优参数
                    if score > best_score:
                        best_score = score
                        best_parameters = {
                            'random_state': rs,
                            'learning_rate': lr,
                            'n_estimators': n_est,
                            'max_depth': depth
                        }
    
    print(f"\n数据量 {ratio*100:.0f}% | 最优 R²_test: {best_score:.5f}")
    print("最优参数:", best_parameters)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_subset, y_subset, test_size=0.2, random_state=best_parameters['random_state']
    )
    sc = StandardScaler()
    X_tr_std = sc.fit_transform(X_tr)
    X_te_std = sc.transform(X_te)
    
    model = GradientBoostingRegressor(
        learning_rate=best_parameters['learning_rate'],
        n_estimators=best_parameters['n_estimators'],
        max_depth=best_parameters['max_depth'],
        random_state=0
    )
    model.fit(X_tr_std, y_tr)
    
    y_tr_pred = model.predict(X_tr_std)
    y_te_pred = model.predict(X_te_std)
    
    r2_train = r2_score(y_tr, y_tr_pred)
    mse_train = mean_squared_error(y_tr, y_tr_pred)
    mae_train = mean_absolute_error(y_tr, y_tr_pred)
    
    r2_test = r2_score(y_te, y_te_pred)
    mse_test = mean_squared_error(y_te, y_te_pred)
    mae_test = mean_absolute_error(y_te, y_te_pred)
    
    print(
        f"训练样本: {X_tr.shape[0]} | "
        f"R²_train: {r2_train:.3f} | MSE_train: {mse_train:.4f} | MAE_train: {mae_train:.4f} | "
        f"R²_test: {r2_test:.3f} | MSE_test: {mse_test:.4f} | MAE_test: {mae_test:.4f}"
    )

print("="*50)


In [ ]:
###特征扰动

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
df=pd.read_csv('data.csv')
X=df.iloc[:,0:49]
y=df.iloc[:,49]

y = df.iloc[:,49].values.ravel()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# ======================
# 2. 标准化
# ======================
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std  = sc.transform(X_test)

# ======================
# 3. 训练基线模型（不扰动）
# ======================
model =GradientBoostingRegressor(n_estimators=91,max_depth=7,random_state=0,alpha=0.1)
model.fit(X_train_std, y_train)

# ======================
# 4. 基线性能
# ======================
y_test_pred = model.predict(X_test_std)
r2_base = r2_score(y_test, y_test_pred)
mae_base = mean_absolute_error(y_test, y_test_pred)
mse_base = mean_squared_error(y_test, y_test_pred)

print("Baseline performance")
print(f"R² = {r2_base:.4f} | MAE = {mae_base:.4f} | MSE = {mse_base:.4f}")


key_features = ['pieceVSA_EState2','pieceHallKierAlpha','piecePEOE_VSA9','pieceMinEStateIndex'] 
print("\nKey features (manually selected):", key_features)


X_test_perturbed = X_test.copy()
noise_ratio = 0.05  

for col in key_features:
    noise = np.random.normal(
        0,
        noise_ratio * X_test[col].std(),
        size=len(X_test)
    )
    X_test_perturbed[col] += noise


X_test_perturbed_std = sc.transform(X_test_perturbed)


y_test_pred_perturbed = model.predict(X_test_perturbed_std)
r2_perturbed = r2_score(y_test, y_test_pred_perturbed)
mae_perturbed = mean_absolute_error(y_test, y_test_pred_perturbed)
mse_perturbed = mean_squared_error(y_test, y_test_pred_perturbed)

print("\nAfter perturbing key features")
print(f"R² = {r2_perturbed:.4f} | MAE = {mae_perturbed:.4f} | MSE = {mse_perturbed:.4f}")

print("\nPerformance degradation")
print(f"ΔR²  = {r2_base - r2_perturbed:.4f}")
print(f"ΔMAE = {mae_perturbed - mae_base:.4f}")
print(f"ΔMSE = {mse_perturbed - mse_base:.4f}")


In [ ]:
###稳定性评估

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



results = []

for random_state in range(100):
    
    X_train, X_test = train_test_split(X, test_size=0.2, random_state=random_state)
    y_train, y_test = train_test_split(y, test_size=0.2, random_state=random_state)

    # 数据标准化
    sc = StandardScaler()
    X_train_std = sc.fit_transform(X_train)
    X_test_std = sc.transform(X_test)

    # 创建模型并训练
    model =GradientBoostingRegressor(random_state=0,n_estimators=91,max_depth =5)
    model.fit(X_train_std, y_train)

    # 预测
    y_train_pred = model.predict(X_train_std)
    y_test_pred = model.predict(X_test_std)

    # 计算评估指标
    MSE_train = mean_squared_error(y_train, y_train_pred)
    MSE_test = mean_squared_error(y_test, y_test_pred)
    MAE_train = mean_absolute_error(y_train, y_train_pred)
    MAE_test = mean_absolute_error(y_test, y_test_pred)
    R2_train = r2_score(y_train, y_train_pred)
    R2_test = r2_score(y_test, y_test_pred)

    # 保存结果
    results.append({
        'random_state': random_state,
        'MSE_train': MSE_train,
        'MSE_test': MSE_test,
        'MAE_train': MAE_train,
        'MAE_test': MAE_test,
        'R2_train': R2_train,
        'R2_test': R2_test
    })

# 将结果转换为 DataFrame 并打印
results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv('100thtsxgb.csv')

In [ ]:
#Fragment analysis

In [ ]:
from rdkit import Chem
from rdkit.Chem import BRICS
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



def get_fragments(
    smiles_list,
    min_frag_size=3,
    max_frag_size=40,
    exclude_small_frags=True,
    repeat_threshold=5
):
    
    all_frags = []
    filtered_frags = []

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            frags = list(BRICS.BRICSDecompose(mol))
            frag_counts = Counter(frags)

            for frag, cnt in frag_counts.items():
                fmol = Chem.MolFromSmiles(frag)
                if not fmol:
                    continue
                n_atoms = fmol.GetNumAtoms()

              
                if n_atoms < min_frag_size and exclude_small_frags:
                    continue
                if n_atoms > max_frag_size:
                    continue

                # === 重复过滤 ===
                if cnt < repeat_threshold:
                    all_frags.extend([frag] * cnt)
                else:
                    filtered_frags.append(frag)

    return Counter(all_frags), set(filtered_frags)



def compute_z_score(fragment, N_high, N_total, count_high, count_total):
    mean = N_high * (count_total / N_total)
    std = np.sqrt(N_high * (count_total / N_total) * (1 - count_total / N_total) * ((N_total - N_high) / (N_total - 1)))
    if std == 0:
        return 0
    return (count_high - mean) / std



def zscore_fragment_analysis(high_perf_smiles, all_smiles, **frag_params):
    high_counts, high_filtered = get_fragments(high_perf_smiles, **frag_params)
    total_counts, total_filtered = get_fragments(all_smiles, **frag_params)

    N_high = len(high_perf_smiles)
    N_total = len(all_smiles)
    z_scores = []

    for frag in total_counts.keys():
        count_high = high_counts.get(frag, 0)
        count_total = total_counts.get(frag, 0)
        z = compute_z_score(frag, N_high, N_total, count_high, count_total)
        is_filtered = frag in high_filtered or frag in total_filtered
        z_scores.append((frag, count_high, count_total, z, is_filtered))

    df = pd.DataFrame(z_scores, columns=["Fragment_SMILES", "Count_in_High", "Count_in_Total", "Z_score", "Filtered"])
    return df.sort_values(by="Z_score", ascending=False)


# === Step 4: 加载数据 ===
def clean_smiles_column(series):
    return series.dropna().astype(str).str.strip()


print("📂 正在读取 fragment.xlsx ...")
data = pd.read_excel("fragment.xlsx")

high_perf_smiles = clean_smiles_column(data.iloc[:, 0])
all_smiles = clean_smiles_column(data.iloc[:, 1])





df_result = zscore_fragment_analysis(high_perf_smiles, all_smiles, **frag_params)

output_file = "fragment_zscore.xlsx"
df_result.to_excel(output_file, index=False)


topN = 20
plt.figure(figsize=(10, 6))
plt.barh(df_result.head(topN)["Fragment_SMILES"], df_result.head(topN)["Z_score"], color="teal")
plt.gca().invert_yaxis()
plt.xlabel("Z-score")
plt.title(f"Top {topN} Enriched Fragments (BRICS Controlled)")
plt.tight_layout()
plt.show()
